## 1) Libraries

In [1]:
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings("ignore")

## 2) Data

In [2]:
# Reading data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [3]:
# Response
y_train = train["host_is_superhost"]

# Drop columns: id, response, and free-text fields not used as features
drop_cols = ["id", "host_is_superhost", "description",
             "host_about", "host_neighbourhood", "host_location"]

X_train = train.drop(columns=drop_cols)
X_test  = test.drop(columns=[c for c in drop_cols if c in test.columns])

In [4]:
# Cleaning

# Strip % and convert to float
for col in ["host_response_rate", "host_acceptance_rate"]:
    X_train[col] = X_train[col].str.rstrip("%").astype(float)
    X_test[col]  = X_test[col].str.rstrip("%").astype(float)

# Extract numeric bathroom count from text (e.g. "1.5 baths" --> 1.5)
for df in [X_train, X_test]:
    df["bathrooms_text"] = (
        df["bathrooms_text"].astype(str).str.extract(r"(\d+\.?\d*)")[0].astype(float)
    )

# Impute missing values
num_cols = X_train.select_dtypes(include="number").columns
cat_cols = X_train.select_dtypes(exclude="number").columns

# Use training mean to impute numeric variables
train_means = X_train[num_cols].mean()
X_train[num_cols] = X_train[num_cols].fillna(train_means)
X_test[num_cols]  = X_test[num_cols].fillna(train_means)

# Add "missing" category for categorical variables
X_train[cat_cols] = X_train[cat_cols].astype(str).fillna("missing")
X_test[cat_cols]  = X_test[cat_cols].astype(str).fillna("missing")

In [5]:
# Label encode categorical columns to use for XGBoost model
for col in cat_cols:
    le = LabelEncoder()
    le.fit(X_train[col])
    X_train[col] = le.transform(X_train[col])
    X_test[col] = X_test[col].apply(lambda x: x if x in le.classes_ else le.classes_[0])
    X_test[col] = le.transform(X_test[col])

## 3) Machine Learning Model

In [6]:
model = XGBClassifier(
    n_estimators = 650,
    learning_rate = 0.06,
    max_depth = 8,
    subsample = 1.0,
    random_state = 16,
    eval_metric = "auc",
    n_jobs = 1
)

model.fit(X_train, y_train)

pred_probs = model.predict_proba(X_test)[:, 1]

## 4) Exporting the Predictions

In [7]:
submission = pd.DataFrame({"id": test["id"], "predicted": pred_probs})
submission.to_csv("submission.csv", index = False)
print(submission.head())

       id  predicted
0  118725   0.983359
1  334671   0.509190
2  407486   0.938409
3  887565   0.995169
4  122507   0.990556
